TODOs:
* Zero-Inflated: 3-pt field goals made, 3-pt field goals attempted, pnr handlers made, pnr handlers attempted, 
* Non-Gaussian: blocks, screen assists, points off screen assists, screens off made, screens off attempted, posts up made, posts up attemtpted, isolations made, isolations attempted, hand offs made, hand offs attempted, cuts made, cuts attempted, pnr handlers made, pnr handlers attempted, pnp made, pnp attempted, opp pick-n-roll shots, opp pick-n-roll shots made, opp pick-n-pop shots, opp pick-n-pop shots made, drives made, drives with shot, right drives, left drives, right drives made, left drives made
*  (FIXED) What's up with 3-pt data? Very low eFG(%) for some players

Requires following packages to be installed with anaconda:
seaborn, pandas, pytorch, openpyxl, jupyter (conda-forge), numpy, sklearn, MissForest

In [3]:
import pandas as pd
import numpy as np
import os
import re

In [5]:
# Minutes in data are displayed e.g. 12:34, which is 12 minutes and 34 seconds. This function converts it to a float
def minute_str_to_float(time):
    minutes = int(time.split(":")[0])
    seconds = int(time.split(":")[1])
    return minutes + seconds / 60

# Same entries are of the form 60%, this function converts it to a float like 0.6
def convert_percentage(percentage):
    if percentage == "-":
        return 0
    return float(percentage.strip("%")) / 100

def load_data() -> pd.DataFrame:
    # Extract League and Season information from filename
    regex = r".+\. (.+) \- (\d{4}-\d{4}).xls$"
    frames = []
    s = 0
    for file in os.listdir("data_raw"):
        filename = os.fsdecode(file)
        m = re.match(regex, filename)
        if m is not None:
            s += 1
            # Test something different
            df = pd.read_excel(os.path.join("data_raw", filename), sheet_name="Box score", engine="openpyxl")
            # Add information contained in filename
            df.insert(0, "Season", [m.group(2)] * len(df), False)
            df.insert(0, "League", [m.group(1)] * len(df), False)
            print(f"Processed {m.group(1)} - {m.group(2)}")
            frames.append(df)
    
    df = pd.concat(frames, ignore_index=True)
    df = df.rename(columns={"Unnamed: 0": "Jersey number", "Unnamed: 1": "Player name", "Unnamed: 2": "Team name"})
    df.columns = df.columns.str.lower()
    # df.replace("-", "0", inplace=True)
     # convert minutes string to float
    df["minutes"] = df["minutes"].apply(minute_str_to_float)
    for column in df.columns.tolist():
        if "%" in column or "percentage" in column:
            df[column] = df[column].apply(convert_percentage)

    return df

# Load data to dataframe
df = load_data()

Processed Liga Endesa - 2020-2021
Processed BETCLIC Elite - LNB Pro A - 2022-2023
Processed Liga Endesa - 2021-2022
Processed Basket League - 2022-2023
Processed LEB Oro - 2020-2021
Processed LEB Oro - 2021-2022
Processed Basketball Bundesliga - 2022-2023
Processed LEB Plata - 2022-2023
Processed FIBA Champions League - 2020-2021
Processed FIBA Champions League - 2021-2022
Processed Basket League - 2020-2021
Processed LEB Oro - 2022-2023
Processed Basket League - 2021-2022
Processed Liga Endesa - 2022-2023
Processed BETCLIC Elite - LNB Pro A - 2021-2022
Processed BETCLIC Elite - LNB Pro A - 2020-2021
Processed FIBA Champions League - 2022-2023
Processed LEB Plata - 2020-2021
Processed Basketball Bundesliga - 2020-2021
Processed LEB Plata - 2021-2022
Processed Basketball Bundesliga - 2021-2022


The below function shows that there are 14 duplicate entries out of a total of about 7400 data points. In addition, most of them didn't play any significant minutes which is why it is reasonable to just drop them.

We observe in a first step that quite many players have played in their national league as well as in one of the European competitions. In a next step we checked if players played in different leagues within the same season. We observe only two players: Sasa Borovnjak and Simas Jarumbauskas. Both playing in the LEB Oro (Spanish first league).

After these observations we need to aggregate the stats of these players by a weighted average (weighted by minutes played).

In [6]:
# Check for duplicate entries for columns "player name", "team name", "league", "season"
# Returns a boolean series indicating if the row is a duplicate
duplicates_indicator = df.duplicated(subset=["player name", "team name", "league", "season"], keep=False)
duplicates_all = df[duplicates_indicator].sort_values(by=["player name", "team name", "league", "season"])
# duplicates = df[duplicates_indicator][["player name", "team name", "league", "season"]].sort_values(by=["player name", "team name", "league", "season"])

# Drop duplicates and perform sanity check
print(f"Num duplicates before drop: {duplicates_indicator.sum()}")
df = df[duplicates_indicator == False]
print(f"Sanity check for num of duplicates:  {df.duplicated(subset=["player name", "team name", "league", "season"], 
                                                            keep=False).sum()}")

Num duplicates before drop: 14
Sanity check for num of duplicates:  0


We consider players playing in multiples seasons as different data points. Also, we don't want to consider the lower (Spanish) leagues for our clustering algorithms. To performs some exploratory analysis we drop some of the meta information like jersey number, team name, season, player name. We proceed by performing some univariate analysis and creating some plots to investigate the relationship between the features. 

In [7]:
# If a player appears in multiple dataframes, aggregate the data, i.e. weighted average by seconds played

df_check_ind = df.duplicated(subset=["player name", "season"], keep=False)
df_mulitiples = df[df_check_ind]
print(f"Check for multiple entries (before): {df_mulitiples.shape[0]}")


def aggregate_stats(df, df_multiples):
    # After aggregating a player should not appear multiple times within the same season
    # Iterate over each player, season pair
    # df.replace("-", np.nan, inplace=True)
    df.replace("-", 0, inplace=True)
    df.replace(np.nan, 0, inplace=True)

    # This will produce a warning "treating keys as positions deprecated". But it isn't an issue because we are using
    # the index simply as a loop counter.
    for i, (player_name, season) in enumerate(df_multiples[["player name", "season"]].drop_duplicates().values):
        # Extract all rows for the player, season pair
        df_player = df[(df["player name"] == player_name) & (df["season"] == season)]
        # First compute minutes player per season
        total_minutes = df_player["minutes"] * df_player["games played"]
        total_across_all = total_minutes.sum()
        weights = total_minutes / total_across_all
        df_player_new = df_player.copy()
        # Not sure if the below line of code is still necessary
        df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].apply(pd.to_numeric, errors='coerce')
        # Multiply each columns by the weights and sum them up. Apply the weights row-wise to indicate
        # the importance of different statistics (weighed by minutes played)
        df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
        # We now have the aggregated data for the player, season pair. Each row contains the the aggregated dat.
        # So we need to drop duplicates.
        df_player_new = df_player_new.drop_duplicates(subset=["player name", "season"])
        # Still need adjust the games played and minutes played. Also because the leage and team data are now ambiguous, 
        # we need to set them to "aggregated".
        total_games = df_player["games played"].sum()
        avg_minutes = df_player["minutes"].mul(pd.Series(df_player["games played"]), axis=0).sum() / total_games
        df_player_new["games played"] = total_games
        df_player_new["minutes"] = avg_minutes
        df_player_new["league"] = "aggregated"
        df_player_new["team name"] = "aggregated"

        # Now, df_player_new contains the aggregated data for the player, season pair (only one row).
        # Need to copy it back to the original dataframe and drop the rows that were aggregated
        df.drop(df[(df["player name"] == player_name) & (df["season"] == season)].index, inplace=True)
        # Append the aggregated data
        df = pd.concat([df, df_player_new], ignore_index=True)
        length = len(df_mulitiples[["player name", "season"]].drop_duplicates())
        if i % 50 == 0:
            print(f"Aggregated data for {i}/{length} players.")

    return df

df = aggregate_stats(df, df_mulitiples)

# Perform sanity check, i.e. check if there are still duplicates. That is, there should only exist one player, season pair
df_check_ind = df.duplicated(subset=["player name", "season"], keep=False)
df_mulitiples = df[df_check_ind]
print(f"Check for multiple entries (after): {df_mulitiples.shape[0]}")

Check for multiple entries (before): 1678
Aggregated data for 0/818 players.


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys a

Aggregated data for 50/818 players.


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys a

Aggregated data for 100/818 players.


...

/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys a

Aggregated data for 800/818 players.


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys a

Check for multiple entries (after): 0


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_28057/2153378488.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)


We now drop some of the data for analysis. We observed that the statistic for "deflections" has not been gathered for every league. In addition, the "usage percentage" is zero for all players. Hence, we remove these two features. We also drop every data point of players which do not play in a first league, and as well players which have not played a sufficient amount. To this end, we define a cutoff of a minimum amount of minutes a player must have played in a specific season.

In [30]:
CUTOFF = 230

print(f"Data points before: {len(df)}")

df_pre_transform = df.copy()
df_pre_transform = df_pre_transform.drop(labels=["deflections", "usage percentage"], axis=1)
# Keep old df to compare to reduced one. Check if the below transformations worked!

def fix_effective_field_goal(df):
    # Assumes that FGA is non-zero
    # eFG percentage is computed as (FG + 0.5 * 3P) / FGA
    # First, find data where eFG is equals to zero
    zero_efg_ind = df['effective field goal percentage'] == 0
    # Ensure that we don't divide by zero
    zero_efg_df = df[zero_efg_ind & (df['field goals attempted'] != 0)].copy()
    # Update orignal df with the new eFG values, but only where eFG is zero
    df.loc[zero_efg_ind, 'effective field goal percentage'] = (zero_efg_df['field goals made'] + 0.5 * zero_efg_df['3-pt field goals made']) / zero_efg_df['field goals attempted']
    # For sanity check: print out number of zeros in effecitve field goal column
    print(f"Number of zeros in eFG column: {df.loc[df['effective field goal percentage'] == 0, "effective field goal percentage"].sum()}")


def drop_too_little_data(df, cutoff):
    """
    df is the dataframe containing all the data over multiple seasons and multiple leagues.
    cutoff is the minimum number of minutes a player must have played in order to be considered for analysis.
    """
    # Drop rows of players having played less than e.g. 230 minutes, i.e. games played times seconds per game
    df.drop(df[df["games played"] * df["minutes"] < cutoff].index, inplace=True)
 
    
def store_normalization_csv(df):
    minutes = pd.Series(df["minutes"])
    num_possesions = pd.Series(df["number of player's possessions"])
    minutes.to_csv("minutes.csv", index=False)
    num_possesions.to_csv("num_possessions.csv", index=False)


low_leagues = ["LEB Plata", "Liga Endesa"]
def drop_low_leagues(df):
    # Drop all rows, i.e. players, where the league is either "LEB Plata" or "Liga Endesa"
    mask = -df["league"].isin(low_leagues)
    df = df[mask]

drop_low_leagues(df_pre_transform)
drop_too_little_data(df_pre_transform, CUTOFF)
# Call this function after the above to make sure we remove data where FG attempted is 0. Would lead to division by 0.
fix_effective_field_goal(df_pre_transform)
store_normalization_csv(df_pre_transform)

Data points before: 6564
Number of zeros in eFG column: 0.0


We now deal with the features which don't seem to follow a Gaussian distribution. These were already listed above but are here for convenience again:

blocks, screen assists, points off screen assists, screens off made, screens off attempted, posts up made, posts up attemtpted, isolations made, isolations attempted, hand offs made, hand offs attempted, cuts made, cuts attempted, pnr handlers made, pnr handlers attempted, pnp made, pnp attempted, opp pick-n-roll shots, opp pick-n-roll shots made, opp pick-n-pop shots, opp pick-n-pop shots made, drives made, drives with shot, right drives, left drives, right drives made, left drives made

We have two normalization candidates: by "minutes" or by "number of player's possessions". We decided on the following:
* For offensive data: normalize by possessions
* For other data: normalize by minutes

However, data points which inherintly already include the number of possessions in their metric should not be normalized. These include:
* 'points per possession', 'offensive rating', 'true shooting percentage' (indirectly)

### Justification for Using Yeo-Johnson Transformation (ChatGPT)

The Yeo-Johnson transformation is applied to several features in this dataset that exhibit exponential-like distributions. This transformation is essential for the following reasons:

1. **Normalize Skewness**: Many machine learning algorithms, including clustering techniques like k-means, assume that the features follow a normal distribution. The Yeo-Johnson transformation helps in correcting skewness and stabilizing the variance of the data, making these assumptions more tenable.

2. **Handle Zero and Negative Values**: Unlike other transformations that require strictly positive values (e.g., Box-Cox), the Yeo-Johnson transformation can be applied to data with zero or negative values, increasing its applicability to our dataset without the need for data adjustments or exclusions.

3. **Enhance Clustering Efficacy**: By transforming skewed features to resemble a normal distribution, the transformed features contribute more effectively to the Euclidean distance calculations used in k-means clustering. This is critical as skewed data can distort distance measures and lead to suboptimal clustering results.

4. **Uniformity and Comparability**: Transforming the features ensures that they are on a more comparable scale, which is crucial when combining multiple features in clustering algorithms. This uniformity can lead to more meaningful and interpretable clusters.

By applying the Yeo-Johnson transformation, we prepare the data in a way that enhances the overall performance of the subsequent clustering process and ensures that the feature scales do not unduly influence the model's outcomes.


In [31]:
from sklearn.preprocessing import PowerTransformer

NORMALIZATION_METHOD = "number of player's possessions"
NON_GAUSSIAN = ['blocks', 'screen assist', 'points off screen assists', 'screens off made', 'screens off attempted',
              'posts up made', 'posts up attempted', 'isolations made', 'isolations attempted', 'hand off made',
              'hand off attempted', 'cuts made', 'cuts attempted', 'pnr handlers made', 'pnr handlers attempted',
              'pnp made', 'pnp attempted', 'opp pick-n-roll shots', 'opp pick-n-roll shots made', 'opp pick-n-pop shots',
              'opp pick-n-pop shots made', 'drives made', 'drives with shot', 'right drives', 'left drives', 
              'right drives made', 'left drives made'

]

# Normalize the data by minutes played before or after applying the log-transformation? Probably apply transformation first.
# Also, better to normalize by number of player possesions or by minutes played?
# 

def apply_log_transformation(df, features):
    """
    Assumption: data for low leagues already dropped and data for players with insuficient minutes played also dropped.
    df: dataframe containing all the data
    features: list of features to be log-transformed
    """
    # add 1 to avoid log(0)
    for feature in features:
        df[feature] = np.log(df[feature] + 1)

def apply_power_tranform(df, features):
    pt = PowerTransformer(method='yeo-johnson')
    for feature in features:
        # Use double brackets because fit_transform expects a 2D array
        df[feature] = pt.fit_transform(df[[feature]])


def drop_irrelevant_features_and_normalize(df, normalization="minutes"):
    """
    normalize: either "minutes", or "number of player's possessions", or None
    """
    # Normalize all columns by minutes played
    minutes = pd.Series(df["minutes"])
    num_possesions = pd.Series(df["number of player's possessions"])
    df.drop(axis=1, labels=df.columns[0:7].tolist(), inplace=True)
    if normalization is not None:
        print(f"Normalized data by {normalization}.")
        # Assumes that we have no zero entries.
        if normalization == "minutes":
            # TODO: think about which features should not be normalized by minutes
            df = df.div(minutes, axis=0)
        else:
            # Normalize all features except "offensive rating", "true shooting percentage", and "points per player's possessions". 
            # These should be kept as they are.
            keep = ["offensive rating", "true shooting percentage", "points per player's possession"]
            df_temp = df.copy()
            df_temp = df_temp.drop(labels=keep, axis=1)
            # Normalize all columns by number of player's possessions
            df_temp = df_temp.div(num_possesions, axis=0)
            # Add dropped columns back to the dataframe by creating a mask for features that were normalized and overriding it with df_temp
            mask = ~df.columns.isin(keep)
            df.loc[:, mask] = df_temp
        return df
    else:
        print("No normalization performed.")
        return df

# df_post_transform = apply_log_transformation(df_pre_transform, NON_GAUSSIAN)
df_post_transform = apply_power_tranform(df_pre_transform, NON_GAUSSIAN)
df_new = drop_irrelevant_features_and_normalize(df_pre_transform, normalization=NORMALIZATION_METHOD)
# Export to csv file s.t. it can be later used for the ML
if NORMALIZATION_METHOD is None:
    df_new.to_csv(f'preprocessed_data.csv', index=False)
else:
    df_new.to_csv(f"preprocessed_normalized_{NORMALIZATION_METHOD}.csv", index=False)
print(f"Data points left: {len(df_new)}")

Normalized data by number of player's possessions.
   offensive rating  true shooting percentage  points per player's possession
0             104.0                      0.61                            1.03
1             102.0                      0.56                            0.94
2             102.0                      0.63                            1.08
3             105.0                      0.65                            1.10
4             104.0                      0.70                            1.07
     points  points per player's possession  field goals made  \
0  0.192857                            1.03          0.082143   
1  0.152632                            0.94          0.057895   
2  0.187500                            1.08          0.071875   
3  0.212766                            1.10          0.074468   
4  0.166667                            1.07          0.056970   

   field goals attempted  field goals, %  3-pt field goals made  \
0               0.13928